In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

In [2]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv("dataset/dataset_updated.csv")

# =========================
# LABEL
# 1 = legitimate, 0 = phishing
# =========================
y = df["ClassLabel"]


In [3]:
# =========================
# FEATURE ENGINEERING
# =========================
from feature_extraction import extract_features

X = []
for url in df["URL"]:
    X.append(extract_features(url))

feature_names = [
    "url_length","has_ip_address","dot_count","https_flag","url_entropy",
    "token_count","subdomain_count","query_param_count",
    "tld_length","path_length","has_hyphen_in_domain",
    "number_of_digits","tld_popularity","suspicious_file_extension",
    "domain_name_length","percentage_numeric_chars",
    "has_phishing_keyword","is_trusted_domain"
]

X = pd.DataFrame(X, columns=feature_names)

In [5]:
print(df["ClassLabel"].isnull().sum())

1


In [6]:
df = df.dropna(subset=["ClassLabel"])

In [7]:
df["ClassLabel"] = pd.to_numeric(df["ClassLabel"], errors="coerce")
df = df.dropna(subset=["ClassLabel"])
df["ClassLabel"] = df["ClassLabel"].astype(int)

In [8]:
X = []
y = df["ClassLabel"]

# safety check
print("NaN label:", y.isnull().sum())

NaN label: 0


In [9]:
# =========================
# CLEAN DATA (IMPORTANT)
# =========================
df = df.dropna(subset=["ClassLabel"])
df = df.dropna(subset=["URL"])

df["ClassLabel"] = pd.to_numeric(df["ClassLabel"], errors="coerce")
df = df.dropna(subset=["ClassLabel"])
df["ClassLabel"] = df["ClassLabel"].astype(int)

y = df["ClassLabel"]

In [12]:
df = df.dropna(subset=["URL", "ClassLabel"])
df = df.reset_index(drop=True)

In [13]:
X = []
y = []

for i, url in enumerate(df["URL"]):
    try:
        feat = extract_features(url)

        # validasi fitur tidak None
        if feat is None:
            continue

        X.append(feat)
        y.append(df["ClassLabel"].iloc[i])

    except Exception as e:
        print(f"Error URL: {url} -> {e}")

In [14]:
y = pd.Series(y)

In [15]:
print("X length:", len(X))
print("y length:", len(y))

X length: 101342
y length: 101342


In [16]:
X = pd.DataFrame(X, columns=feature_names)

In [17]:
# =========================
# SPLIT DATA (IMPORTANT)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
# =========================================================
# 1. RANDOM FOREST (MAIN MODEL - CONTROL OVERFITTING)
# =========================================================
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,              # penting untuk hindari overfitting
    min_samples_leaf=5,       # penting
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=5, n_estimators=200,
                       random_state=42)

In [19]:
# =========================================================
# 2. DECISION TREE (BASELINE)
# =========================================================
dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=5,
    random_state=42
)

dt.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42)

In [20]:
# =========================================================
# 3. LOGISTIC REGRESSION (LINEAR BASELINE)
# =========================================================
lr = LogisticRegression(
    max_iter=1000
)

lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [22]:
# =========================
# EVALUATION FUNCTION
# =========================
def evaluate(model, name):
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    print(f"\n===== {name} =====")
    print("Train Accuracy:", train_acc)
    print("Test Accuracy:", test_acc)
    print(classification_report(y_test, model.predict(X_test)))

In [48]:
# =========================
# EVALUATE ALL MODELS
# =========================
evaluate(rf, "Random Forest")
evaluate(dt, "Decision Tree")
evaluate(lr, "Logistic Regression")



===== Random Forest =====
Train Accuracy: 0.998766543732192
Test Accuracy: 0.9983225615471903
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12743
           1       1.00      1.00      1.00      7526

    accuracy                           1.00     20269
   macro avg       1.00      1.00      1.00     20269
weighted avg       1.00      1.00      1.00     20269


===== Decision Tree =====
Train Accuracy: 0.998618528980055
Test Accuracy: 0.9973851694706202
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12743
           1       1.00      1.00      1.00      7526

    accuracy                           1.00     20269
   macro avg       1.00      1.00      1.00     20269
weighted avg       1.00      1.00      1.00     20269


===== Logistic Regression =====
Train Accuracy: 0.9940794099145215
Test Accuracy: 0.994671666091075
              precision    recall  f1-score   support



In [49]:
# =========================
# SAVE BEST MODEL (RF biasanya terbaik)
# =========================
joblib.dump(rf, "model/rf_modelnewtrainbgt.pkl")
joblib.dump(dt, "model/dt_modelnewtrainbgt.pkl")
joblib.dump(lr, "model/lr_modelnewtrainbgt.pkl")

print("\nModels saved successfully!")


Models saved successfully!


In [35]:
import pandas as pd

importances = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print(importances)

path_length                  0.275134
token_count                  0.206954
https_flag                   0.135723
percentage_numeric_chars     0.109666
number_of_digits             0.089740
dot_count                    0.084043
subdomain_count              0.024763
has_ip_address               0.020504
tld_popularity               0.015141
tld_length                   0.011756
url_length                   0.011120
url_entropy                  0.006238
suspicious_file_extension    0.004154
domain_name_length           0.004030
has_hyphen_in_domain         0.000403
query_param_count            0.000297
is_trusted_domain            0.000262
has_phishing_keyword         0.000073
dtype: float64


In [25]:
print(df["ClassLabel"].value_counts())

ClassLabel
0    63713
1    37629
Name: count, dtype: int64


In [37]:
RandomForestClassifier(
    max_depth=8,
    min_samples_leaf=10,
    max_features="sqrt"
)

RandomForestClassifier(max_depth=8, min_samples_leaf=10)

In [38]:
is_trusted_domain = X["is_trusted_domain"].iloc[0]
has_phishing_keyword = X["has_phishing_keyword"].iloc[0]

if is_trusted_domain == 1:
    risk_score -= 0.3

if has_phishing_keyword == 1:
    risk_score += 0.5

In [39]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

In [45]:
import pandas as pd

importances = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print(importances)

path_length                  0.275134
token_count                  0.206954
https_flag                   0.135723
percentage_numeric_chars     0.109666
number_of_digits             0.089740
dot_count                    0.084043
subdomain_count              0.024763
has_ip_address               0.020504
tld_popularity               0.015141
tld_length                   0.011756
url_length                   0.011120
url_entropy                  0.006238
suspicious_file_extension    0.004154
domain_name_length           0.004030
has_hyphen_in_domain         0.000403
query_param_count            0.000297
is_trusted_domain            0.000262
has_phishing_keyword         0.000073
dtype: float64


In [43]:
from urllib.parse import urlparse

# contoh URL
url = "https://google.com/login/test"

# =========================
# PARSE URL (WAJIB ADA)
# =========================
parsed = urlparse(url)

url_parts = {
    "scheme": parsed.scheme,
    "domain": parsed.netloc,
    "path": parsed.path,
    "query": parsed.query
}

# =========================
# DOMAIN LOGIC
# =========================
domain = url_parts["domain"].lower()

trusted_domains = [
    "google.com",
    "youtube.com",
    "github.com",
    "facebook.com"
]

suspicious_brands = ["login", "secure", "verify", "account"]

trusted = any(domain.endswith(d) for d in trusted_domains)
suspicious_brand = any(b in domain for b in suspicious_brands)

# =========================
# DOMAIN SCORE
# =========================
domain_score = 0

if trusted:
    domain_score = 1.0

elif suspicious_brand:
    domain_score = 0.8

else:
    domain_score = 0.3

print("domain:", domain)
print("trusted:", trusted)
print("score:", domain_score)

domain: google.com
trusted: True
score: 1.0


In [44]:
def analyze_url(url):
    parsed = urlparse(url)

    domain = parsed.netloc.lower()

    trusted_domains = ["google.com", "github.com"]
    suspicious = ["login", "verify"]

    trusted = any(domain.endswith(d) for d in trusted_domains)
    suspicious_brand = any(b in domain for b in suspicious)

    return {
        "domain": domain,
        "trusted": trusted,
        "suspicious": suspicious_brand
    }

In [46]:
def process_url(url, model, explainer, feature_names):

    # =========================
    # 1. PARSE URL
    # =========================
    parsed = urlparse(url)
    domain = parsed.netloc.lower()

    url_parts = {
        "domain": domain,
        "path": parsed.path,
        "query": parsed.query
    }

    # =========================
    # 2. FEATURE EXTRACTION
    # =========================
    features = extract_features(url)
    X = pd.DataFrame([features], columns=feature_names)

    # =========================
    # 3. PREDICTION
    # =========================
    proba = model.predict_proba(X)[0]
    phishing_prob = proba[1]
    risk_score = phishing_prob * 100

    # =========================
    # 4. CONTEXT CHECK
    # =========================
    trusted_domains = ["google.com", "github.com", "youtube.com"]
    suspicious_keywords = ["login", "verify", "secure", "account"]

    trusted = any(domain.endswith(d) for d in trusted_domains)
    suspicious = any(k in domain for k in suspicious_keywords)

    if trusted:
        risk_score *= 0.5
        label = "Legitimate"
    elif suspicious:
        risk_score = max(risk_score, 80)
        label = "Phishing"
    else:
        label = "Suspicious"

    # =========================
    # 5. SHAP (EXPLANATION)
    # =========================
    shap_values = explainer.shap_values(X)
    shap_values = shap_values[1][0]

    top_idx = sorted(
        range(len(shap_values)),
        key=lambda i: abs(shap_values[i]),
        reverse=True
    )[:5]

    explanations = [
        f"{feature_names[i]} berkontribusi pada keputusan"
        for i in top_idx
    ]

    # =========================
    # 6. RETURN RESULT
    # =========================
    return {
        "url": url,
        "domain": domain,
        "label": label,
        "risk_score": round(risk_score, 2),
        "explanations": explanations
    }

In [47]:
@app.route("/", methods=["GET", "POST"])
def index():
    result = None

    if request.method == "POST":
        url = request.form["url"]

        result = process_url(
            url,
            model,
            explainer,
            feature_names
        )

    return render_template("index.html", result=result)

NameError: name 'app' is not defined